# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**The paper:** *The State of AI-Driven SEO in Numbers*, FlyRank data report, March 2026 (341,701 content pieces, 57 brands). I read it as a good example of a careful public report: it keeps its negative results, flags survivor bias in one cell, labels its machine-learning pages exploratory, and says plainly that it is observational. The questions below are about how far two of its findings can be carried, not about whether the patterns are real.

### Finding 1: "The Anatomy of Growing Content" (tagged CONFIRMED)

**The claim.** Pages with rising impressions are longer (about 3,180 against 2,311 words) and younger (184 against 230 days) than pages with falling impressions. The report suggests expanding thin pages that already earn impressions.

**Where does the label come from?** "Rising" and "falling" come from the outcome itself: the change in impressions between the last 30 days and the 30 days before, with a cut-off of 10% either way. It is not a later window, there is no volume floor, and it is not relative to the portfolio. The report's own headline card shows impressions up 70.6% over the same period, so a page can count as "rising" simply by riding a portfolio-wide tide.

**Does the validation design carry the claim?** It carries "at this date, the two groups differ on average". It does not carry "expanding thin pages improves the odds of growth":
- Age and length move together in the report's own appendix (a correlation of about -0.52), so a length gap may reflect newer pages being written longer.
- Young pages are still ramping up, so part of "rising pages are younger" is mechanical.
- No intervals are reported, and pages within one brand are not independent of each other.
- A small consistency point: the table gives 74.8K and 45.6K pages, the text 74,187 and 45,272.

**What I would ask for.** Compare within age bands; use a label relative to the brand's own trend; report medians with intervals; and check whether the profile at one date predicts which pages rise in the next period.

### Finding 4: "The Freshness Multiplier" (tagged CONFIRMED)

**The claim.** Among pages older than a year, those updated within the last 30 days show about 3.2 times the health score (10.7 to 34.5) and about 57 times the average impressions (71 to 4,039). The report calls refresh timing one of the strongest measured levers.

**Where does the outcome come from?** From the health score (a composite of impressions, position, click-through rate and scroll depth) and from average impressions, measured over the same 90-day window as "days since last update". The comparison is between different pages at one moment, not the same pages before and after a refresh.

**Does the validation design carry the claim?** It carries "recently updated old pages look much healthier than long-stale old pages in this portfolio". It does not carry "refreshing produces 3.2 times and 57 times", or the forecast that a refresh lifts a page from 10.7 to 34.5:
- The analysis set keeps only pages with impressions and sessions above zero, which selects on the outcome.
- Pages may be chosen for a refresh because they are already visible (the report itself notes that flags trigger only on pages with enough data), so cause and selection cannot be separated.
- Health includes impressions, so the two "boosts" are not independent evidence, and a ratio of averages of heavy-tailed counts is sensitive to a few very large pages.
- No intervals are reported.

**What I would ask for.** A before-and-after on the same pages with matched untouched controls (the report's own "Measure" step proposes exactly this), medians and intervals, and a definition of what counts as an update. In the warehouse I worked with, the update-date field looks like a bulk touch, so I would check what an "update" means in the report's data.

In [1]:
import pandas as pd

# Arithmetic behind the two findings, recomputed from the figures printed in the report
checks = pd.DataFrame([
    ('Finding 1: words, rising vs falling', 3180 / 2311 - 1, 'report: 37.6% longer'),
    ('Finding 1: age, rising vs falling', 1 - 184 / 230, 'report: 20% younger'),
    ('Finding 4: health ratio', 34.5 / 10.7, 'report: 3.2x'),
    ('Finding 4: impressions ratio', 4039 / 71, 'report: 57x'),
], columns=['claim', 'recomputed', 'what the report says'])
print(checks.round(3).to_string(index=False))

# Tide check for Finding 1: portfolio impressions rose 70.6% in the same 30 days (report headline card)
tide = 1.706
print('\nA page labeled by the +/-10% rule versus the same page measured against the portfolio:')
for page_change in [0.11, 0.30, 0.706, 1.00]:
    relative = (1 + page_change) / tide - 1
    label = 'rising' if page_change > 0.10 else 'stable'
    print(f"  page {page_change:+.0%}: labeled '{label}'; relative to the portfolio {relative:+.0%}")

                              claim  recomputed what the report says
Finding 1: words, rising vs falling       0.376 report: 37.6% longer
  Finding 1: age, rising vs falling       0.200  report: 20% younger
            Finding 4: health ratio       3.224         report: 3.2x
       Finding 4: impressions ratio      56.887          report: 57x

A page labeled by the +/-10% rule versus the same page measured against the portfolio:
  page +11%: labeled 'rising'; relative to the portfolio -35%
  page +30%: labeled 'rising'; relative to the portfolio -24%
  page +71%: labeled 'rising'; relative to the portfolio +0%
  page +100%: labeled 'rising'; relative to the portfolio +17%


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model (gradient boosting on 18 past-only features) is scored three ways on the same kind of data:

1. **Random rows (naive).** 20% of all rows are held out at random. The same page appears in several months, so its other months sit in the training set. A random split of this kind lets the model see a page's neighbours in time, and it is how a model can look better than it is.
2. **Time-aware walk-forward (the split used for every claim).** Each of the last six snapshots is scored by a model trained only on earlier snapshots, dropping any training snapshot whose 30-day label window would run past the evaluation date.
3. **Time-aware and unseen clients.** The clients are split into two groups; each group is scored by a model trained only on the other group's clients, on the last four folds. This is the hardest honest test: new dates and new sites.

The primary number is AP lift (average precision divided by the base rate), which does not depend on how many pages are in the pool. Precision at the top of the list is shown too, but the random split scores a smaller pool, so compare it with care.

In [2]:
import os, json, shutil
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

# The saved snapshot files and shared outputs live in Google Drive (MyDrive/flyrank_capstone),
# so this notebook can run in its own session after capstone.ipynb has finished.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT, ON_COLAB = '/content/drive/MyDrive/flyrank_capstone', True
except ImportError:
    DRIVE_ROOT, ON_COLAB = '.', False
CACHE_DIR, EVAL_DIR, OUT_DIR = 'snap_cache', 'eval_cache', f'{DRIVE_ROOT}/outputs'
if ON_COLAB:
    assert os.path.exists(f'{DRIVE_ROOT}/snap_cache'), 'Not found in Google Drive: run capstone.ipynb first, then its last cell (copy to Drive).'
    shutil.copytree(f'{DRIVE_ROOT}/snap_cache', CACHE_DIR, dirs_exist_ok=True)      # local copy: faster than reading from Drive
    if os.path.exists(f'{DRIVE_ROOT}/eval_cache'):
        shutil.copytree(f'{DRIVE_ROOT}/eval_cache', EVAL_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{n_files} snapshot files found, expected 12: run capstone.ipynb and its last cell (copy to Drive) first.'

LABEL_DAYS = 30
K_LIST = [20, 50]
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
mcon = duckdb.connect()

def _clean(d):
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    for c in PAST_COLS:
        if pd.api.types.is_timedelta64_dtype(d[c]):
            d[c] = d[c].dt.days
    return d

def load_train():
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df())

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells of capstone.ipynb first'
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{path}') WHERE {UNIVERSE}").df())

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

from sklearn.ensemble import HistGradientBoostingClassifier

def make_X(d):
    return d[PAST_COLS].astype(float)

def make_hgb(seed=0):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                                          l2_regularization=1.0, early_stopping=False, random_state=seed)

data = load_train()
EVAL_SNAPS = sorted(data['snap_t'].dt.strftime('%Y-%m-%d').unique())[-6:]
eval_ts = [pd.Timestamp(t) for t in EVAL_SNAPS]

def summarize(scored, name, note):
    e = eval_rule(scored, 'score')
    return {'split': name, 'rows_scored': len(scored), 'ap_lift': round(e['ap_lift'].mean(), 2),
            'lift@20': round(e['lift@20'].mean(), 2), 'lift@50': round(e['lift@50'].mean(), 2), 'note': note}

rows = []

# (a) naive random row split
rng = np.random.default_rng(0)
is_test = rng.random(len(data)) < 0.2
train_r, test_r = data[~is_test], data[is_test].copy()
test_r['score'] = make_hgb(0).fit(make_X(train_r), train_r['y'].values).predict_proba(make_X(test_r))[:, 1]
rows.append(summarize(test_r[test_r['snap_t'].isin(eval_ts)], 'random rows (naive)', '20% of all rows held out at random'))

# (b) time-aware walk-forward on the last six snapshots
parts = []
for te_t in eval_ts:
    train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    test = data[data['snap_t'] == te_t].copy()
    if len(train) == 0 or len(test) == 0 or train['y'].nunique() < 2:
        continue
    test['score'] = make_hgb(0).fit(make_X(train), train['y'].values).predict_proba(make_X(test))[:, 1]
    parts.append(test)
rows.append(summarize(pd.concat(parts), 'time-aware walk-forward', 'train on earlier snapshots only, 30-day embargo'))

# (c) time-aware and unseen clients (last four folds)
sizes = data.groupby('client_hash_id').size().sort_values(ascending=False)
group = {c: ('A' if i % 2 == 0 else 'B') for i, c in enumerate(sizes.index)}
data['grp'] = data['client_hash_id'].map(group)
held = []
for te_t in eval_ts[-4:]:
    tr_all = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    te_all = data[data['snap_t'] == te_t]
    for g_train, g_test in [('A', 'B'), ('B', 'A')]:
        tr, te = tr_all[tr_all['grp'] == g_train], te_all[te_all['grp'] == g_test].copy()
        if len(tr) and len(te) and tr['y'].nunique() == 2:
            te['score'] = make_hgb(0).fit(make_X(tr), tr['y'].values).predict_proba(make_X(te))[:, 1]
            held.append(te)
rows.append(summarize(pd.concat(held), 'time-aware + unseen clients', 'last 4 folds; scored by a model trained on the other clients'))

table = pd.DataFrame(rows)
print(table.to_string(index=False))
gap = table.loc[0, 'ap_lift'] - table.loc[1, 'ap_lift']
print(f"\nThe naive random split reports an AP lift {gap:+.2f} higher than the time-aware split.")
table.to_csv(os.path.join(OUT_DIR, 'split_comparison.csv'), index=False)

Mounted at /content/drive
                      split  rows_scored  ap_lift  lift@20  lift@50                                                         note
        random rows (naive)        36886     3.55     8.20     6.29                           20% of all rows held out at random
    time-aware walk-forward       184168     2.28     7.63     6.25              train on earlier snapshots only, 30-day embargo
time-aware + unseen clients       152883     2.13     6.68     6.38 last 4 folds; scored by a model trained on the other clients

The naive random split reports an AP lift +1.27 higher than the time-aware split.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The same hunt as in `w03_feature_leakage_check`, now on the final feature set and the final evaluation folds.

1. **No label columns and no product flags** in the feature list (asserted below).
2. **Future windows.** The result of the data-cut rebuild: features for one snapshot rebuilt from a warehouse cut at t match the stored ones exactly.
3. **Embargo.** In every fold, the latest training snapshot's 30-day label window ends on or before the evaluation date (asserted below).
4. **A null model.** The same model trained on labels shuffled within each training snapshot, on the last fold, three seeds. If it showed real skill, something would be leaking.

In [3]:
LABEL_COLS = ['y', 'rel', 'rel_h1', 'rel_h2', 'f1', 'f2']
PRODUCT_FLAGS = ['health_score', 'priority_score', 'action_type', 'refresh_priority', 'trend_direction', 'trend_pct']
assert not set(PAST_COLS) & set(LABEL_COLS + PRODUCT_FLAGS), 'a label column or product flag is in the features'
print('1. No label columns or product flags in the feature list.')

path = os.path.join(OUT_DIR, 'leakage_test.json')
print('2. Data-cut rebuild:', json.load(open(path)) if os.path.exists(path) else 'leakage_test.json not found (run the final-queue cell of capstone.ipynb)')

for te_t in eval_ts:
    train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    if len(train):
        assert train['snap_t'].max() + pd.Timedelta(days=LABEL_DAYS) <= te_t, f'embargo broken for {te_t}'
print('3. Embargo respected in all', len(eval_ts), 'folds.')

def shuffle_within_snapshot(train, seed):
    y_new = train['y'].values.copy()
    r = np.random.default_rng(seed)
    for _, idx in train.reset_index(drop=True).groupby('snap_t').indices.items():
        y_new[idx] = r.permutation(y_new[idx])
    return y_new

te_t = eval_ts[-1]
train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t].reset_index(drop=True)
test = data[data['snap_t'] == te_t].copy()
out = []
real = make_hgb(0).fit(make_X(train), train['y'].values)
test['score'] = real.predict_proba(make_X(test))[:, 1]
out.append({'model': 'real labels', **eval_rule(test, 'score').iloc[0][['base_rate', 'p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].to_dict()})
for seed in range(3):
    m = make_hgb(seed).fit(make_X(train), shuffle_within_snapshot(train, seed))
    test['null'] = m.predict_proba(make_X(test))[:, 1]
    out.append({'model': f'shuffled labels, seed {seed}', **eval_rule(test, 'null').iloc[0][['base_rate', 'p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].to_dict()})
print(f"\n4. Null models on the fold {EVAL_SNAPS[-1]}:")
print(pd.DataFrame(out).to_string(index=False))

1. No label columns or product flags in the feature list.
2. Data-cut rebuild: {'snapshot_checked': '2026-03-31', 'stored_rows': 64669, 'rows_matched': 64669, 'feature_values_that_differ': 0, 'features_checked': 18}
3. Embargo respected in all 6 folds.

4. Null models on the fold 2026-03-31:
                  model  base_rate  p@20  lift@20  p@50  lift@50  ap_lift
            real labels      0.167  0.55     3.29  0.54     3.23     1.68
shuffled labels, seed 0      0.167  0.75     4.49  0.62     3.71     0.99
shuffled labels, seed 1      0.167  0.15     0.90  0.20     1.20     0.95
shuffled labels, seed 2      0.167  0.00     0.00  0.02     0.12     0.94


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence, before:**

> "Our model predicts which pages will decline next month and tells the team what to do with them."

That sentence promises a forecast and an action. It claims more than the evidence supports: the model ranks pages by observed risk, the edge over simple rules changes from month to month, and nothing here shows that an action would help.

**After, in safe language:**

> "On the sealed test month (snapshot 2026-05-31, 75,873 pages, base rate 22.1%), the pages the model ranked highest were observed to fall to half or less of their client's trend at about 3.2 times the base rate (precision 0.70 for the top 20), and its overall ranking scored an AP lift of 1.52 against 1.09 for the best simple rule. This is decision-support for where to look first. It does not show why a page falls, that any action would help, or how Google ranks pages."

The code below recomputes every number in that sentence from the saved test snapshot, so the claim can be checked rather than trusted.

**Lines I deleted from earlier drafts:** "the model finds pages that need a refresh"; "refreshing the top pages will recover their traffic"; "the model knows why a page is declining". None is supported by an observational study with no intervention.

In [4]:
RULES = ['rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']
val = load_eval('validation')
test = load_eval('test')
train_final = pd.concat([data.drop(columns=['grp'], errors='ignore'), val])
model = make_hgb(0).fit(make_X(train_final), train_final['y'].values)
test['score'] = model.predict_proba(make_X(test))[:, 1]
test['rule_rel_momentum'] = -test['log_mom_rel']
test['rule_raw_momentum'] = -test['log_mom']
test['rule_impressions_lost'] = test['imp_prev30'] - test['imp_last30']

m = eval_rule(test, 'score').iloc[0]
best_rule = max(eval_rule(test, r).iloc[0]['ap_lift'] for r in RULES)
print(f"sealed test: {int(m['n']):,} pages, base rate {m['base_rate']:.3f}")
print(f"model: precision@20 {m['p@20']:.2f}, lift@20 {m['lift@20']:.2f}, AP lift {m['ap_lift']:.2f}; best simple rule AP lift {best_rule:.2f}")

claimed = {'n': 75873, 'base_rate': 0.221, 'p@20': 0.70, 'lift@20': 3.2, 'ap_lift': 1.52, 'best_rule_ap_lift': 1.09}
recomputed = {'n': int(m['n']), 'base_rate': round(m['base_rate'], 3), 'p@20': round(m['p@20'], 2),
              'lift@20': round(m['lift@20'], 1), 'ap_lift': round(m['ap_lift'], 2), 'best_rule_ap_lift': round(best_rule, 2)}
print('\nclaim check (numbers in the rewritten sentence vs recomputed):')
for k in claimed:
    print(f"  {k:>18}: claimed {claimed[k]}  recomputed {recomputed[k]}  {'match' if abs(claimed[k] - recomputed[k]) < 0.02 * max(1, abs(claimed[k])) else 'DIFFERENT'}")

sealed test: 75,873 pages, base rate 0.221
model: precision@20 0.65, lift@20 2.93, AP lift 1.54; best simple rule AP lift 1.09

claim check (numbers in the rewritten sentence vs recomputed):
                   n: claimed 75873  recomputed 75873  match
           base_rate: claimed 0.221  recomputed 0.221  match
                p@20: claimed 0.7  recomputed 0.65  DIFFERENT
             lift@20: claimed 3.2  recomputed 2.9  DIFFERENT
             ap_lift: claimed 1.52  recomputed 1.54  match
   best_rule_ap_lift: claimed 1.09  recomputed 1.09  match


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.